# Day 58 — AI Agents
### ReAct pattern, tool use, memory, LangGraph stateful agents

## 1. Setup

In [1]:
import anthropic
import json
import re

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

print("Anthropic client ready")
print("Model:", MODEL)

Anthropic client ready
Model: claude-haiku-4-5


## 2. The ReAct Pattern — Manual Implementation

In [2]:
# define tools the agent can use
tools = [
    {
        "name": "calculator",
        "description": "Perform mathematical calculations. Input should be a mathematical expression as a string.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Math expression to evaluate e.g. '2 + 2' or '15 * 7'"}
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_word_count",
        "description": "Count the number of words in a given text.",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "Text to count words in"}
            },
            "required": ["text"]
        }
    },
    {
        "name": "text_uppercase",
        "description": "Convert text to uppercase.",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "Text to convert to uppercase"}
            },
            "required": ["text"]
        }
    }
]

def run_tool(tool_name, tool_input):
    if tool_name == "calculator":
        try:
            result = eval(tool_input["expression"])
            return str(result)
        except Exception as e:
            return f"Error: {e}"
    elif tool_name == "get_word_count":
        return str(len(tool_input["text"].split()))
    elif tool_name == "text_uppercase":
        return tool_input["text"].upper()
    return "Unknown tool"

print("Tools defined:", [t["name"] for t in tools])

Tools defined: ['calculator', 'get_word_count', 'text_uppercase']


## 3. The ReAct Loop — Reasoning + Acting

In [3]:
def run_react_agent(task, max_steps=5):
    messages = [{"role": "user", "content": task}]
    step = 0

    print(f"TASK: {task}")
    print("=" * 60)

    while step < max_steps:
        step += 1
        print(f"\nStep {step}:")

        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            tools=tools,
            messages=messages
        )

        # check if agent wants to use a tool
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  THINK: I need to use {block.name}")
                    print(f"  ACT:   {block.name}({block.input})")

                    result = run_tool(block.name, block.input)
                    print(f"  OBS:   Result = {result}")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

        # agent has finished
        elif response.stop_reason == "end_turn":
            final = response.content[0].text
            print(f"\nFINAL ANSWER: {final}")
            return final

    return "Max steps reached"

result = run_react_agent("What is 347 multiplied by 28? Then tell me how many words are in the answer when written out.")

TASK: What is 347 multiplied by 28? Then tell me how many words are in the answer when written out.

Step 1:
  THINK: I need to use calculator
  ACT:   calculator({'expression': '347 * 28'})
  OBS:   Result = 9716

Step 2:
  THINK: I need to use get_word_count
  ACT:   get_word_count({'text': 'nine thousand seven hundred sixteen'})
  OBS:   Result = 5

Step 3:

FINAL ANSWER: **Answer:**
- 347 × 28 = **9,716**
- When written out as "nine thousand seven hundred sixteen", it contains **5 words**


## 4. LangGraph — Stateful Agent

In [4]:
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated
import operator

# define tools as LangChain tools
@tool
def calculator(expression: str) -> str:
    """Perform mathematical calculations. Input a math expression like '2 + 2'."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@tool
def get_word_count(text: str) -> str:
    """Count the number of words in a given text."""
    return str(len(text.split()))

@tool
def text_uppercase(text: str) -> str:
    """Convert text to uppercase."""
    return text.upper()

lc_tools = [calculator, get_word_count, text_uppercase]

llm = ChatAnthropic(model="claude-haiku-4-5", max_tokens=500)
llm_with_tools = llm.bind_tools(lc_tools)

print("LangGraph tools registered:", [t.name for t in lc_tools])

LangGraph tools registered: ['calculator', 'get_word_count', 'text_uppercase']


## 5. Define the LangGraph State and Nodes

In [5]:
# define the agent state
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

# node 1: the agent (LLM decides what to do)
def agent_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# node 2: the tool executor
def tool_node(state: AgentState):
    last_message = state["messages"][-1]
    tool_results = []
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        for t in lc_tools:
            if t.name == tool_name:
                result = t.invoke(tool_args)
                tool_results.append(
                    ToolMessage(content=str(result), tool_call_id=tool_call["id"])
                )
    return {"messages": tool_results}

# routing: should we call a tool or are we done?
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# build the graph
graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

app = graph.compile()
print("LangGraph agent compiled successfully")

LangGraph agent compiled successfully


## 6. Run the LangGraph Agent

In [6]:
def run_langgraph_agent(task):
    print(f"TASK: {task}")
    print("=" * 60)

    initial_state = {"messages": [HumanMessage(content=task)]}
    final_state = app.invoke(initial_state)

    print("\nMESSAGE TRACE:")
    for msg in final_state["messages"]:
        msg_type = type(msg).__name__
        if msg_type == "HumanMessage":
            print(f"  Human:   {msg.content[:80]}")
        elif msg_type == "AIMessage":
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  AI Tool: {tc['name']}({tc['args']})")
            else:
                print(f"  AI:      {msg.content[:120]}")
        elif msg_type == "ToolMessage":
            print(f"  Result:  {msg.content}")

    final_answer = final_state["messages"][-1].content
    print(f"\nFINAL ANSWER: {final_answer}")
    return final_answer

run_langgraph_agent("Calculate 125 * 16, then convert the number 'two thousand' to uppercase.")

TASK: Calculate 125 * 16, then convert the number 'two thousand' to uppercase.

MESSAGE TRACE:
  Human:   Calculate 125 * 16, then convert the number 'two thousand' to uppercase.
  AI Tool: calculator({'expression': '125 * 16'})
  AI Tool: text_uppercase({'text': 'two thousand'})
  Result:  2000
  Result:  TWO THOUSAND
  AI:      Perfect! Here are the results:

1. **125 × 16 = 2000**
2. **'two thousand' in uppercase = 'TWO THOUSAND'**

Interestingl

FINAL ANSWER: Perfect! Here are the results:

1. **125 × 16 = 2000**
2. **'two thousand' in uppercase = 'TWO THOUSAND'**

Interestingly, the calculation result (2000) is the same as the number spelled out in the text!


"Perfect! Here are the results:\n\n1. **125 × 16 = 2000**\n2. **'two thousand' in uppercase = 'TWO THOUSAND'**\n\nInterestingly, the calculation result (2000) is the same as the number spelled out in the text!"

## 7. Adding Memory — In-Context Conversation History

In [7]:
def run_agent_with_memory(conversation_history, new_message):
    conversation_history.append(HumanMessage(content=new_message))

    state = {"messages": conversation_history}
    final_state = app.invoke(state)

    # update history with new messages from this turn
    conversation_history.extend(final_state["messages"][len(conversation_history):])

    last_ai = [m for m in final_state["messages"] if type(m).__name__ == "AIMessage"][-1]
    return last_ai.content, conversation_history

# start a fresh conversation
history = []

print("Turn 1:")
answer, history = run_agent_with_memory(history, "What is 84 * 12?")
print("Answer:", answer[:150])
print(f"History length: {len(history)} messages\n")

print("Turn 2:")
answer, history = run_agent_with_memory(history, "Now add 500 to that result.")
print("Answer:", answer[:150])
print(f"History length: {len(history)} messages\n")

print("Turn 3:")
answer, history = run_agent_with_memory(history, "What was my first calculation?")
print("Answer:", answer[:150])

Turn 1:
Answer: 84 * 12 = **1008**
History length: 4 messages

Turn 2:
Answer: 1008 + 500 = **1508**
History length: 8 messages

Turn 3:
Answer: Your first calculation was **84 * 12 = 1008**.


## 8. Summary — What We Built Today

| Component | What it does | Key concept |
|-----------|-------------|-------------|
| ReAct loop (manual) | Reasoning + Acting + Observing cycle | Agent decides tool, runs it, observes result, repeats |
| Tool definition | JSON schema describing what each tool does | Model reads description to decide when to call it |
| LangGraph StateGraph | Defines agent as a graph of nodes + edges | State flows between agent node and tool node |
| AgentState | TypedDict holding conversation messages | Shared state across all graph nodes |
| Conditional edges | should_continue() routes to tools or END | Graph decides next step based on agent output |
| Parallel tool calls | LangGraph calls multiple tools in one step | More efficient than sequential when tasks are independent |
| In-context memory | Full conversation history sent every turn | Agent resolves 'that result' and 'my first calculation' correctly |

**The ReAct loop in plain English:**
1. THINK: what do I need to do next?
2. ACT: call a tool with specific inputs
3. OBSERVE: read the tool's output
4. Repeat until task is complete, then give final answer

**Why LangGraph over manual ReAct:**
Manual ReAct works but has no state management, no graph structure,
and no built-in support for parallel tool calls or complex branching.
LangGraph handles all of this while keeping the loop logic explicit and debuggable.